In [1]:
%pip install -q ultralytics scikit-learn pyyaml tqdm pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 6.0 MB/s eta 0:00:00


In [1]:
import sys, os, re, math, copy, json, time, shutil
from pathlib import Path
import numpy as np, pandas as pd, torch, yaml, ultralytics

print("PyTorch:", torch.__version__, "| Ultralytics:", ultralytics.__version__)
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
PyTorch: 2.11.0+cu130 | Ultralytics: 8.4.173
GPU: Tesla T4 14.6 GB


In [2]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DATASET = Path("/content/drive/MyDrive/chenab_yolov12_cpu_paper_reproduction")
DRIVE_ROOT    = Path("/content/drive/MyDrive/fabric_b3_v2")      # code + runs + results persist here
LOCAL_DATASET = Path("/content/dataset")                          # fast local copy used for training

RUNS_DIR, RESULTS_DIR, CFG_DIR = DRIVE_ROOT/"runs", DRIVE_ROOT/"results", DRIVE_ROOT/"cfg"
for p in (RUNS_DIR, RESULTS_DIR, CFG_DIR): p.mkdir(parents=True, exist_ok=True)

if not LOCAL_DATASET.exists():
    print("Copying dataset to local disk (a few minutes the first time)...")
    shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)

orig = yaml.safe_load((LOCAL_DATASET/"data.yaml").read_text())
names = orig["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names, key=int)]
NUM_CLASSES = len(names)

COLAB_YAML = CFG_DIR/"data_local.yaml"
COLAB_YAML.write_text(yaml.safe_dump({
    "path": str(LOCAL_DATASET), "train": "train/images", "val": "valid/images",
    "test": "test/images", "nc": NUM_CLASSES, "names": names}, sort_keys=False))
print(NUM_CLASSES, "classes:", names)

Mounted at /content/drive
Copying dataset to local disk (a few minutes the first time)...
7 classes: ['baekra', 'color issues', 'contamination', 'cut', 'gray stitch', 'selvet', 'stain']


## Dataset audit
Checks label validity, class balance, **train/val/test leakage** (Roboflow augmentations of the same source image landing in different splits inflate scores) and defect size — which tells you whether a larger `imgsz` will help more than any architecture change.

In [3]:
from PIL import Image
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def audit(split):
    d = LOCAL_DATASET/split
    imgs = sorted(p for p in (d/"images").iterdir() if p.suffix.lower() in IMG_EXT)
    counts = np.zeros(NUM_CLASSES, int); boxes = []
    issues = dict(no_label_file=0, empty_label=0, bad_rows=0)
    for im in imgs:
        lf = d/"labels"/f"{im.stem}.txt"
        if not lf.exists():
            issues["no_label_file"] += 1; continue
        rows = [l.split() for l in lf.read_text().splitlines() if l.strip()]
        if not rows: issues["empty_label"] += 1
        for r in rows:
            try:
                c = int(float(r[0])); w, h = float(r[3]), float(r[4])
                assert len(r) == 5 and 0 <= c < NUM_CLASSES and 0 < w <= 1 and 0 < h <= 1
            except Exception:
                issues["bad_rows"] += 1; continue
            counts[c] += 1; boxes.append((w, h))
    return imgs, counts, np.array(boxes), issues

AUDIT = {s: audit(s) for s in ("train", "valid", "test")}
for s, (imgs, counts, boxes, issues) in AUDIT.items():
    print(f"{s:6s} images={len(imgs):5d} boxes={counts.sum():6d} {issues}")

cls_df = pd.DataFrame({s: AUDIT[s][1] for s in AUDIT}, index=names)
display(cls_df)
tc = AUDIT["train"][1]
print(f"Class imbalance (max/min train instances): {tc.max()/max(tc.min(),1):.1f}x")

# leakage between splits (Roboflow: <name>_jpg.rf.<hash>.jpg)
base = lambda p: re.sub(r"_(jpg|jpeg|png|bmp)\.rf\.[0-9a-f]+$", "", p.stem, flags=re.I)
B = {s: {base(p) for p in AUDIT[s][0]} for s in AUDIT}
for a, b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
    n = len(B[a] & B[b])
    print(f"shared source images {a}/{b}: {n}", "  <-- LEAKAGE, re-split by source image!" if n else "")

# defect size at different input resolutions
imgs, _, boxes, _ = AUDIT["train"]
W, H = np.median([Image.open(p).size for p in imgs[:200]], axis=0)
print(f"\nMedian image size: {W:.0f}x{H:.0f}")
for s in (640, 800, 960, 1280):
    k = s / max(W, H)
    px = np.sqrt(boxes[:, 0]*W*k * boxes[:, 1]*H*k)
    print(f"imgsz={s}: median defect {np.median(px):6.1f}px | <16px: {(px<16).mean()*100:5.1f}% | <32px: {(px<32).mean()*100:5.1f}%")

train  images= 1950 boxes=  2708 {'no_label_file': 0, 'empty_label': 161, 'bad_rows': 1}
valid  images=  562 boxes=   786 {'no_label_file': 0, 'empty_label': 38, 'bad_rows': 0}
test   images=  280 boxes=   368 {'no_label_file': 0, 'empty_label': 19, 'bad_rows': 0}


,train,valid,test
baekra,245,79,24
color issues,232,30,21
contamination,169,55,17
cut,346,102,47
gray stitch,333,87,48
selvet,421,121,58
stain,962,312,153


Class imbalance (max/min train instances): 5.7x
shared source images train/valid: 335   <-- LEAKAGE, re-split by source image!
shared source images train/test: 188   <-- LEAKAGE, re-split by source image!
shared source images valid/test: 106   <-- LEAKAGE, re-split by source image!

Median image size: 1122x1952
imgsz=640: median defect  129.6px | <16px:   0.0% | <32px:   3.5%
imgsz=800: median defect  162.1px | <16px:   0.0% | <32px:   1.2%
imgsz=960: median defect  194.5px | <16px:   0.0% | <32px:   0.4%
imgsz=1280: median defect  259.3px | <16px:   0.0% | <32px:   0.0%


## ECA module + model builder
ECA is saved as an importable file (so checkpoints can be re-loaded later) and registered into Ultralytics' parser, so it can be used directly in the YAML.
**Re-run this cell and the next in any new session before loading a checkpoint.**

In [4]:
ECA_SRC = """
import torch
import torch.nn as nn

class ECA(nn.Module):
    # Efficient Channel Attention (Wang et al., CVPR 2020): GAP -> 1D conv across channels -> sigmoid gate
    def __init__(self, k=3):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=int(k), padding=int(k) // 2, bias=False)

    def forward(self, x):
        y = self.pool(x).squeeze(-1).transpose(-1, -2)      # B,C,1,1 -> B,1,C
        y = self.conv(y).transpose(-1, -2).unsqueeze(-1)     # B,1,C -> B,C,1,1
        return x * torch.sigmoid(y)
"""
(DRIVE_ROOT/"eca_module.py").write_text(ECA_SRC)
if str(DRIVE_ROOT) not in sys.path: sys.path.insert(0, str(DRIVE_ROOT))

import importlib, eca_module
importlib.reload(eca_module)
import ultralytics.nn.tasks as tasks
tasks.ECA = eca_module.ECA          # lets parse_model resolve "ECA" in YAML
print("ECA registered.")

ECA registered.


In [6]:
from ultralytics import YOLO
from ultralytics.nn.tasks import DetectionModel

SCALE = "n"
OFFICIAL = yaml.safe_load((Path(ultralytics.__file__).parent/"cfg/models/12/yolo12.yaml").read_text())

def eca_k(c):
    t = int(abs((math.log2(c) + 1) / 2))
    return t if t % 2 else t + 1

def build_variant(name, p2, eca, scale=SCALE):
    """YAML from the official YOLO12 file. Backbone (0-8) and head layers 9-14 are untouched.
    p2  : add stride-4 branch (4 detection scales).   eca : ECA on every detection feature map."""
    bb, hd = copy.deepcopy(OFFICIAL["backbone"]), copy.deepcopy(OFFICIAL["head"])
    assert len(bb) == 9 and hd[-1][2] == "Detect" and hd[5][2] == "A2C2f", "official yaml layout changed"
    _, width, max_ch = OFFICIAL["scales"][scale]

    if p2:
        head = hd[:6] + [                                        # layers 9-14 = official top-down path
            [-1, 1, "nn.Upsample", [None, 2, "nearest"]],        # 15
            [[-1, 2], 1, "Concat", [1]],                         # 16  P2 skip = backbone layer 2 (C3k2 @ stride 4)
            [-1, 2, "A2C2f", [256, False, -1]],                  # 17  P2 out (Increased from 128 to 256 to ensure hidden channels are multiple of 32)
            [-1, 1, "Conv", [128, 3, 2]],                        # 18
            [[-1, 14], 1, "Concat", [1]],                        # 19
            [-1, 2, "A2C2f", [512, False, -1]],                  # 20  P3 out (Increased from 256 to 512 to ensure hidden channels are multiple of 32)
            [-1, 1, "Conv", [256, 3, 2]],                        # 21  == official 15
            [[-1, 11], 1, "Concat", [1]],                        # 22
            [-1, 2, "A2C2f", [512, False, -1]],                  # 23  P4 out == official 17
            [-1, 1, "Conv", [512, 3, 2]],                        # 24  == official 18
            [[-1, 8], 1, "Concat", [1]],                         # 25
            [-1, 2, "C3k2", [1024, True]],                       # 26  P5 out == official 20
        ]
        outs = [17, 20, 23, 26]
        pre = {i: i for i in range(15)} | {21: 15, 23: 17, 24: 18, 26: 20}   # new layer -> official layer
    else:
        head, outs = hd[:-1], [14, 17, 20]
        pre = {i: i for i in range(21)}

    n_layers = len(bb) + len(head)
    detect_from = outs
    if eca:
        allL = bb + head
        extra = []
        for o in outs:
            c = math.ceil(min(allL[o][3][0], max_ch) * width / 8) * 8
            extra.append([o, 1, "ECA", [eca_k(c)]])
        head += extra
        detect_from = list(range(n_layers, n_layers + len(outs)))
    head.append([detect_from, 1, "Detect", ["nc"]])

    cfg = {"nc": NUM_CLASSES, "scales": OFFICIAL["scales"], "backbone": bb, "head": head}
    path = CFG_DIR/f"yolo12{scale}-{name}.yaml"
    path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return path, pre

def make_init_checkpoint(name, p2, eca):
    cfg_path, pre = build_variant(name, p2, eca)
    net = DetectionModel(str(cfg_path), nc=NUM_CLASSES, verbose=False)
    off = YOLO(f"yolo12{SCALE}.pt").model.state_dict()

    sd, copied = net.state_dict(), {}
    for k, v in sd.items():
        parts = k.split(".")
        if parts[0] != "model": continue
        i = int(parts[1])
        if i in pre:
            ok = ".".join(["model", str(pre[i])] + parts[2:])
            if ok in off and off[ok].shape == v.shape:
                sd[k] = off[ok].clone(); copied[i] = copied.get(i, 0) + 1
    net.load_state_dict(sd)

    total = sum(k.startswith("model.") for k in sd)
    n_eca = sum(isinstance(m, eca_module.ECA) for m in net.modules())
    det = net.model[-1]
    print(f"[{name}] pretrained tensors transferred: {sum(copied.values())}/{total} | "
          f"detection scales: {len(det.f)} (strides {[int(s) for s in det.stride]}) | ECA modules: {n_eca}")
    assert n_eca == (len(det.f) if eca else 0), "ECA missing from model!"

    init = CFG_DIR/f"{name}_init.pt"
    torch.save({"model": net, "train_args": {}}, init)
    return init

# --- build and sanity-check B3 ---
init_b3 = make_init_checkpoint("B3_eca_p2", p2=True, eca=True)
m = YOLO(str(init_b3)); m.info()
with torch.no_grad():
    out = m.model.eval()(torch.zeros(1, 3, 640, 640))
print("Forward pass OK")

[B3_eca_p2] pretrained tensors transferred: 569/849 | detection scales: 4 (strides [4, 8, 16, 32]) | ECA modules: 4
YOLO12n-B3_eca_p2 summary: 335 layers, 2,843,294 parameters, 2,843,278 gradients, 14.9 GFLOPs
Forward pass OK


## Training recipe (identical for A / B1 / B2 / B3)
Set `IMG_SIZE` from the audit above: if many defects are < 32 px at 640, use 800–960.

In [9]:
IMG_SIZE = 640            # try 800 or 960 if the audit shows many small defects

TRAIN_CFG = dict(
    imgsz=IMG_SIZE, epochs=100, patience=8, batch=16, device=0,
    workers=min(8, os.cpu_count() or 2),
    optimizer="AdamW", lr0=0.001, lrf=0.01, weight_decay=0.0005, cos_lr=True,
    warmup_epochs=3.0,
    # colour kept mild (stains/dyes are colour-defined); geometry fabric-friendly
    hsv_h=0.015, hsv_s=0.45, hsv_v=0.35,
    degrees=3.0, translate=0.10, scale=0.30, shear=2.0, perspective=0.0003,
    fliplr=0.5, flipud=0.5,                 # line defects keep their orientation under flips
    mosaic=1.0, close_mosaic=15, mixup=0.0, # strong aug early, clean fine-tuning at the end
    amp=True, cache="disk", seed=42, deterministic=True,
    project=str(RUNS_DIR), exist_ok=True, plots=True, save=True, verbose=True,
)
VARIANTS = {                     # name : (P2, ECA)
    "A_yolo12n": (False, False),
    "B1_eca":    (False, True),
    "B2_p2":     (True,  False),
    "B3_eca_p2": (True,  True),
}

def train_variant(name):
    p2, eca = VARIANTS[name]
    init = make_init_checkpoint(name, p2, eca)
    model = YOLO(str(init))
    model.train(data=str(COLAB_YAML), name=name, **TRAIN_CFG)
    return Path(model.trainer.best)

## Train
Check the log's first lines: `Transferred N/M items from pretrained weights` and that the model summary lists the ECA layers.
If Colab disconnects: `YOLO(RUNS_DIR/"B3_eca_p2"/"weights"/"last.pt").train(resume=True)` (after re-running the ECA cells).

In [10]:
# B3 = proposed model. Add "A_yolo12n", "B1_eca", "B2_p2" to rerun the whole ablation under the SAME recipe.
RUN = ["B3_eca_p2"]
for n in RUN:
    print("=" * 60, "\nTRAINING", n, "\n" + "=" * 60)
    best = train_variant(n)
    print("best:", best)

TRAINING B3_eca_p2 
[B3_eca_p2] pretrained tensors transferred: 569/849 | detection scales: 4 (strides [4, 8, 16, 32]) | ECA modules: 4
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/fabric_b3_v2/cfg/data_local.yaml, degrees=3.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.45, hsv_v=0.35, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, 

## Evaluation
Inference settings (`imgsz`, NMS `iou`, TTA) are chosen on **val**; **test** is then evaluated once.

In [11]:
def evaluate(pt, split, imgsz=None, iou=0.7, augment=False, plots=False):
    r = YOLO(str(pt)).val(data=str(COLAB_YAML), split=split, imgsz=imgsz or IMG_SIZE, batch=16,
                          iou=iou, augment=augment, conf=0.001, device=0, plots=plots, verbose=False)
    p, rc = float(r.box.mp), float(r.box.mr)
    return dict(P=p*100, R=rc*100, F1=(2*p*rc/(p+rc) if p+rc else 0)*100,
                mAP50=float(r.box.map50)*100, mAP50_95=float(r.box.map)*100), r

def sweep_val(pt):
    rows = []
    for imgsz in sorted({IMG_SIZE, IMG_SIZE + 160}):
        for iou in (0.5, 0.6, 0.7):
            for tta in (False, True):
                try:
                    s, _ = evaluate(pt, "val", imgsz, iou, tta)
                    rows.append(dict(imgsz=imgsz, iou=iou, tta=tta, **s))
                except Exception as e:
                    print("skipped", imgsz, iou, tta, type(e).__name__)
    return pd.DataFrame(rows).sort_values("mAP50_95", ascending=False).reset_index(drop=True)

BEST_B3 = RUNS_DIR/"B3_eca_p2"/"weights"/"best.pt"
assert BEST_B3.exists(), BEST_B3

# confirm the trained checkpoint really contains ECA
chk = YOLO(str(BEST_B3))
print("ECA modules in trained B3:", sum(isinstance(x, eca_module.ECA) for x in chk.model.modules()))

sw = sweep_val(BEST_B3)
display(sw.round(2))
top = sw.iloc[0]
TUNED = dict(imgsz=int(top.imgsz), iou=float(top.iou), augment=bool(top.tta))
print("Selected on VAL:", TUNED)

ECA modules in trained B3: 4
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO12n-B3_eca_p2 summary (fused): 197 layers, 2,830,822 parameters, 0 gradients, 14.5 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 26.0±12.1 MB/s, size: 73.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/dataset/valid/labels.cache... 562 images, 38 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 562/562 124.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 3.0it/s 11.9s
                   all        562        786      0.802      0.712      0.786      0.439
Speed: 1.6ms preprocess, 10.6ms inference, 0.0ms loss, 2.7ms postprocess per image
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO12n-B3_eca_p2 summary (fused): 197 

,imgsz,iou,tta,P,R,F1,mAP50,mAP50_95
0,800,0.6,True,77.18,73.77,75.44,79.85,45.17
1,800,0.7,True,74.12,72.63,73.37,77.88,45.11
2,640,0.7,False,79.45,71.24,75.12,77.95,45.06
3,800,0.5,True,77.35,74.58,75.94,79.80,44.69
4,640,0.6,False,80.00,71.39,75.45,78.61,44.64
5,640,0.5,False,80.18,71.23,75.44,78.60,43.94
6,640,0.6,True,76.50,72.21,74.29,78.13,42.70
7,640,0.7,True,73.82,71.06,72.41,75.81,42.65
8,640,0.5,True,77.10,72.10,74.51,78.34,42.12
9,800,0.7,False,75.24,67.72,71.28,73.80,38.16


Selected on VAL: {'imgsz': 800, 'iou': 0.6, 'augment': True}


In [12]:
test_default, r_def = evaluate(BEST_B3, "test", IMG_SIZE, 0.7, False, plots=True)
test_tuned, r_tun   = evaluate(BEST_B3, "test", TUNED["imgsz"], TUNED["iou"], TUNED["augment"])
display(pd.DataFrame({"B3 default": test_default, "B3 val-tuned inference": test_tuned}).T.round(2))

b = r_def.box
per_class = pd.DataFrame({"class": [names[i] for i in b.ap_class_index], "P": b.p*100, "R": b.r*100,
                          "AP50": b.ap50*100, "AP50-95": b.ap*100})
per_class["train_instances"] = [int(tc[i]) for i in b.ap_class_index]
display(per_class.sort_values("AP50-95").round(2))   # weakest classes first

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO12n-B3_eca_p2 summary (fused): 197 layers, 2,830,822 parameters, 0 gradients, 14.5 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 38.6±17.7 MB/s, size: 94.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/dataset/test/labels... 280 images, 19 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 280/280 590.7it/s 0.5s
val: New cache created: /content/dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.9it/s 6.1s
                   all        280        368      0.745      0.731      0.798      0.453
Speed: 2.2ms preprocess, 9.4ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to /content/runs/detect/val-13
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CU

,P,R,F1,mAP50,mAP50_95
B3 default,74.55,73.07,73.80,79.85,45.34
B3 val-tuned inference,72.63,76.60,74.56,80.23,45.38


,class,P,R,AP50,AP50-95,train_instances
1,color issues,63.20,57.24,63.86,24.77,232
3,cut,71.30,57.45,70.54,40.02,346
5,selvet,68.45,67.34,72.47,41.10,421
4,gray stitch,78.98,62.50,75.88,46.63,333
6,stain,87.92,89.54,89.40,54.14,962
0,baekra,79.92,83.33,89.08,54.53,245
2,contamination,72.09,94.12,97.69,56.17,169


## Optional: low-LR refinement
`close_mosaic=15` already gives a clean fine-tuning phase, so expect only a small gain here. It is kept optional and the choice between B3 and refined is made on **val**, not test.

In [13]:
RUN_REFINE = False
if RUN_REFINE:
    refine_cfg = {**TRAIN_CFG, **dict(epochs=20, patience=10, lr0=2e-4, warmup_epochs=1.0,
                  mosaic=0.0, close_mosaic=0, scale=0.10, translate=0.03, degrees=2.0, shear=1.0,
                  hsv_s=0.25, hsv_v=0.20)}
    YOLO(str(BEST_B3)).train(data=str(COLAB_YAML), name="B3_refined", **refine_cfg)
    BEST_REF = RUNS_DIR/"B3_refined"/"weights"/"best.pt"
    v_b3, _  = evaluate(BEST_B3, "val", **{k: TUNED[k] for k in ("imgsz", "iou", "augment")})
    v_ref, _ = evaluate(BEST_REF, "val", **{k: TUNED[k] for k in ("imgsz", "iou", "augment")})
    print("VAL mAP50-95  B3:", round(v_b3["mAP50_95"], 2), "| refined:", round(v_ref["mAP50_95"], 2))
    FINAL = BEST_REF if v_ref["mAP50_95"] > v_b3["mAP50_95"] else BEST_B3
else:
    FINAL = BEST_B3
print("FINAL model:", FINAL)

FINAL model: /content/drive/MyDrive/fabric_b3_v2/runs/B3_eca_p2/weights/best.pt


## Final comparison table + efficiency profile
Includes every variant that has been trained under this recipe.

In [16]:
def profile(pt, imgsz):
    m = YOLO(str(pt)); net = m.model.cuda().eval()
    params = sum(p.numel() for p in net.parameters())
    try: flops = m.model.info(verbose=False, imgsz=imgsz)[3]
    except Exception: flops = float("nan")
    x = torch.randn(1, 3, imgsz, imgsz, device="cuda")
    with torch.no_grad(), torch.autocast("cuda"):
        for _ in range(20): net(x)
        torch.cuda.synchronize(); t = time.time()
        for _ in range(100): net(x)
        torch.cuda.synchronize()
    return dict(Params_M=params/1e6, GFLOPs=flops, ms_per_img=(time.time()-t)*10, Size_MB=Path(pt).stat().st_size/2**20)

rows = []
for n in VARIANTS:
    pt = RUNS_DIR/n/"weights"/"best.pt"
    if pt.exists():
        s, _ = evaluate(pt, "test", IMG_SIZE, 0.7, False)
        rows.append({"Model": n, **s, **profile(pt, IMG_SIZE)})
table = pd.DataFrame(rows)
display(table.round(2))

if len(table) > 1 and "A_yolo12n" in set(table.Model):
    a = table[table.Model == "A_yolo12n"].iloc[0]
    delta = table.set_index("Model")[["P", "R", "F1", "mAP50", "mAP50_95"]] - a[["P", "R", "F1", "mAP50", "mAP50_95"]]
    display(delta.round(2).rename_axis("Δ vs A (pp)"))

table.to_csv(RESULTS_DIR/"ablation_test_results.csv", index=False)
json.dump({"final_model": str(FINAL), "test_default": test_default, "test_tuned": test_tuned, "tuned_settings": TUNED,
           "train_cfg": {k: v for k, v in TRAIN_CFG.items() if k != "project"}},
          open(RESULTS_DIR/"B3_final.json", "w"), indent=2)
shutil.copy2(FINAL, DRIVE_ROOT/"ECA_P2_YOLOv12n_best.pt")
print("Saved to", DRIVE_ROOT)

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO12n-B3_eca_p2 summary (fused): 197 layers, 2,830,822 parameters, 0 gradients, 14.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1616.4±272.9 MB/s, size: 52.1 KB)
val: Scanning /content/dataset/test/labels.cache... 280 images, 19 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 280/280 69.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.6it/s 5.0s
                   all        280        368      0.745      0.731      0.798      0.453
Speed: 1.1ms preprocess, 8.9ms inference, 0.0ms loss, 1.9ms postprocess per image


,Model,P,R,F1,mAP50,mAP50_95,Params_M,GFLOPs,ms_per_img,Size_MB
0,B3_eca_p2,74.55,73.07,73.8,79.85,45.34,2.84,NaN,32.95,5.93


Saved to /content/drive/MyDrive/fabric_b3_v2


In [19]:
# ============================================================
# COMPREHENSIVE PERFORMANCE METRICS DISPLAY (DYNAMIC)
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# 1. Resolve available metrics object in memory
eval_metrics = None
for var_name in ["test_metrics", "val_metrics", "validation_results", "metrics"]:
    if var_name in globals() and globals()[var_name] is not None:
        eval_metrics = globals()[var_name]
        print(f"✓ Using metrics from: `{var_name}`")
        break

# If no metrics object exists in memory, evaluate model automatically
if eval_metrics is None:
    target_model = globals().get("best_model") or globals().get("model")
    if target_model is None:
        # Try loading best.pt from runs folder
        runs_path = globals().get("RUNS_DIR", Path("/content/drive/MyDrive/fabric_b3_v2/runs"))
        candidates = list(Path(runs_path).rglob("best.pt"))
        if candidates:
            from ultralytics import YOLO
            best_path = max(candidates, key=lambda p: p.stat().st_mtime)
            print(f"Loading best model from: {best_path}")
            target_model = YOLO(str(best_path))
        else:
            raise RuntimeError("No model or evaluation results found in memory or disk.")

    yaml_path = str(globals().get("COLAB_YAML") or globals().get("COLAB_DATA_YAML") or globals().get("DATA_YAML", "data.yaml"))
    eval_metrics = target_model.val(data=yaml_path, split="test", plots=False)

# 2. Extract Overall Metrics
box = eval_metrics.box
names = eval_metrics.names

p_mean = float(box.mp)
r_mean = float(box.mr)
map50_mean = float(box.map50)
map5095_mean = float(box.map)
f1_mean = (2 * p_mean * r_mean / (p_mean + r_mean)) if (p_mean + r_mean) > 0 else 0.0

# 3. Print Overall Benchmark Summary Card
print("\n" + "=" * 65)
print("             OVERALL PERFORMANCE METRICS (BENCHMARK)             ")
print("=" * 65)
print(f"  • Precision (P)            : {p_mean * 100:6.2f}%")
print(f"  • Recall (R)               : {r_mean * 100:6.2f}%")
print(f"  • F1-Score                 : {f1_mean * 100:6.2f}%")
print(f"  • mAP@0.50                 : {map50_mean * 100:6.2f}%")
print(f"  • mAP@0.50:0.95            : {map5095_mean * 100:6.2f}%")
print("=" * 65 + "\n")

# 4. Compute Dynamic Class-by-Class Table
class_records = []
for i, class_idx in enumerate(box.ap_class_index):
    p_c, r_c, ap50_c, ap_c = box.class_result(i)
    f1_c = (2 * p_c * r_c) / (p_c + r_c) if (p_c + r_c) > 0 else 0.0

    class_records.append({
        "Class": names[class_idx],
        "Precision (%)": round(p_c * 100, 2),
        "Recall (%)": round(r_c * 100, 2),
        "F1-Score (%)": round(f1_c * 100, 2),
        "mAP@0.50 (%)": round(ap50_c * 100, 2),
        "mAP@0.50:0.95 (%)": round(ap_c * 100, 2)
    })

# Add overall summary row
class_records.append({
    "Class": "ALL (Mean)",
    "Precision (%)": round(p_mean * 100, 2),
    "Recall (%)": round(r_mean * 100, 2),
    "F1-Score (%)": round(f1_mean * 100, 2),
    "mAP@0.50 (%)": round(map50_mean * 100, 2),
    "mAP@0.50:0.95 (%)": round(map5095_mean * 100, 2)
})

metrics_df = pd.DataFrame(class_records)

# 5. Display Formatted Markdown / HTML Table
print("DETAILED CLASS-WISE BREAKDOWN:")
display(
    metrics_df.style
    .format({col: "{:.2f}%" for col in metrics_df.columns if col != "Class"})
    .set_properties(**{'text-align': 'center'})
)

# 6. Save as CSV for Paper / Report
output_csv = Path(globals().get("RESULTS_DIR", ".")) / "detailed_performance_metrics.csv"
metrics_df.to_csv(output_csv, index=False)
print(f"\n✓ Saved table to: {output_csv}")

Loading best model from: /content/drive/MyDrive/fabric_b3_v2/runs/B3_eca_p2/weights/best.pt
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO12n-B3_eca_p2 summary (fused): 197 layers, 2,830,822 parameters, 0 gradients, 14.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2397.6±1024.9 MB/s, size: 153.3 KB)
val: Scanning /content/dataset/test/labels.cache... 280 images, 19 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 280/280 97.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.6it/s 6.8s
                   all        280        368      0.745      0.731      0.798      0.453
                baekra         17         24      0.799      0.833      0.891      0.545
          color issues         14         21      0.632      0.572      0.639      0.248
         contamination         14         17      0.721      0.941      0.977      0.562
                   cut       

,Class,Precision (%),Recall (%),F1-Score (%),mAP@0.50 (%),mAP@0.50:0.95 (%)
0,baekra,79.92%,83.33%,81.59%,89.08%,54.53%
1,color issues,63.20%,57.24%,60.07%,63.86%,24.77%
2,contamination,72.09%,94.12%,81.64%,97.69%,56.17%
3,cut,71.30%,57.45%,63.63%,70.54%,40.02%
4,gray stitch,78.98%,62.50%,69.78%,75.88%,46.63%
5,selvet,68.45%,67.34%,67.89%,72.47%,41.10%
6,stain,87.92%,89.54%,88.72%,89.40%,54.14%
7,ALL (Mean),74.55%,73.07%,73.80%,79.85%,45.34%



✓ Saved table to: /content/drive/MyDrive/fabric_b3_v2/results/detailed_performance_metrics.csv


In [15]:
YOLO(str(FINAL)).predict(source=str(LOCAL_DATASET/"test"/"images"), imgsz=TUNED["imgsz"], conf=0.25, iou=TUNED["iou"],
                         device=0, save=True, project=str(RESULTS_DIR), name="B3_predictions", exist_ok=True, verbose=False)
print("Predictions:", RESULTS_DIR/"B3_predictions")

Results saved to /content/drive/MyDrive/fabric_b3_v2/results/B3_predictions
Predictions: /content/drive/MyDrive/fabric_b3_v2/results/B3_predictions


## Reporting rules
* Compare B3 only against A/B1/B2 trained with this same `TRAIN_CFG`. The old PART A numbers (CPU recipe) are not a fair baseline for a GPU-recipe model.
* Report `test_default` in the ablation table; report val-tuned inference or the refined model as a separate "best deployment setting" row.
* For a paper, repeat A and B3 with 3 seeds (`seed=0,1,2`) and report mean ± std; differences under ~0.5 mAP are within noise.
* Largest single accuracy levers to try next (one at a time): `IMG_SIZE` 800/960, scale `"s"` (set `SCALE = "s"`), and class-imbalance handling if the audit shows a large ratio.